In [ ]:
# tamserve-boxes: the motion gate, recording kept by what it holds, and live view from one connection per camera.
# One cell, Colab's CPU runtime: install GStreamer and Rust if needed, build the crate, make the 1-channel model and a test
# clip, measure the gate against the detector (MCC), run a box that records and streams, read it back through the
# OpenAI-compatible endpoint, sweep the recordings, and check the Lean 4 proof of the retention rules.
import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

RUST_GIT = os.environ.get("TSB_RUST_GIT", "https://github.com/dhaneshkk/tamserve-boxes")
BRANCH = os.environ.get("TSB_BRANCH", "main")
WORK = Path("/content" if Path("/content").exists() else Path.cwd()) / "motion_record_live"
WORK.mkdir(parents=True, exist_ok=True)
CRATE = WORK / "tamserve-boxes"
sh = lambda *a, **k: subprocess.run(list(a), check=True, **k)
out = lambda *a: subprocess.run(list(a), check=True, capture_output=True, text=True, env=globals().get("env")).stdout

if not shutil.which("gst-inspect-1.0") or subprocess.run(["gst-inspect-1.0", "x264enc"], capture_output=True).returncode:
    sh("bash", "-c", "sudo apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
       "libgstreamer1.0-dev libgstreamer-plugins-base1.0-dev gstreamer1.0-plugins-base gstreamer1.0-plugins-good "
       "gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly gstreamer1.0-libav gstreamer1.0-tools ffmpeg zstd")
if not CRATE.exists():
    sh("git", "clone", "-q", "--depth", "1", "-b", BRANCH, RUST_GIT, str(CRATE))
sh(sys.executable, "-m", "pip", "install", "-q", "openvino>=2024.0")
cargo = shutil.which("cargo") or str(Path.home() / ".cargo/bin/cargo")
if not Path(cargo).exists():
    sh("bash", "-c", "curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    cargo = str(Path.home() / ".cargo/bin/cargo")

# OpenVINO's C library from the pip wheel, under the plain name the Rust bindings look for.
import openvino
ov = Path(openvino.__file__).parent / "libs"
lib = WORK / "ovlib"; lib.mkdir(exist_ok=True)
c_lib = next(ov.glob("libopenvino_c.so.*"))
if not (lib / "libopenvino_c.so").exists():
    (lib / "libopenvino_c.so").symlink_to(c_lib)
env = dict(os.environ, LD_LIBRARY_PATH=f"{lib}:{ov}:" + os.environ.get("LD_LIBRARY_PATH", ""))
sh(cargo, "build", "--release", "--bins", "--examples", "-q", cwd=CRATE, env=env)
B, MCC = CRATE / "target/release/tamserve-boxes", CRATE / "target/release/examples/gate_mcc"

model, gray = WORK / "yolox_nano.onnx", WORK / "yolox_nano_gray.onnx"
if not model.exists():
    urllib.request.urlretrieve("https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx", model)
sh(str(B), "fold", str(model), str(gray), env=env)

# A clip: 12 s still, 12 s of a moving ball, 6 s still. Use your own night and day footage to tune the gate for real.
clip = WORK / "clip.mp4"
if not clip.exists():
    src = lambda pattern, n: f"videotestsrc pattern={pattern} num-buffers={n} ! video/x-raw,width=640,height=360,framerate=25/1 ! c. "
    sh("bash", "-c", f"gst-launch-1.0 -q concat name=c ! x264enc key-int-max=25 tune=zerolatency ! h264parse ! mp4mux ! "
       f"filesink location={clip} " + src("black", 300) + src("ball", 300) + src("black", 150))

# 1. The gate against the detector: MCC, recall (frames with something on them it would have slept through) and looks saved.
gate = json.loads(out(str(MCC), str(gray), str(clip), "10", "30"))
print("motion gate vs detector:", {k: gate[k] for k in ("tp", "fp", "tn", "fn", "mcc", "recall", "looks_saved")})

# 2. A box that follows with the gate, records and serves live, from one connection to the camera.
rec = WORK / "rec"; shutil.rmtree(rec, ignore_errors=True)
cfg = WORK / "cams.toml"
cfg.write_text(f'''[detector]
model = "{gray}"
threads = 2
classes = ["person", "car", "sports ball"]
[tracker]
look_every = 1
[decode]
decoder = "software"
realtime = true
[http]
listen = "127.0.0.1:9321"
[motion]
hold = 10
[record]
dir = "{rec}"
segment_seconds = 4
motion_days = 3
objects_days = 14
sweep_seconds = 3600
[[camera]]
name = "gate"
url = "{clip}"
record = true
live = true
''')
box = subprocess.Popen([str(B), "run", "--config", str(cfg)], stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, env=env)
try:
    time.sleep(16)
    with urllib.request.urlopen("http://127.0.0.1:9321/live/gate.ts", timeout=10) as r:
        live = r.read(64 * 188)
    ask = json.dumps({"model": "gate", "messages": [{"role": "user", "content": "what does the gate camera show?"}]}).encode()
    req = urllib.request.Request("http://127.0.0.1:9321/v1/chat/completions", ask, {"Content-Type": "application/json"})
    answer = json.load(urllib.request.urlopen(req, timeout=10))
    cam = json.loads(answer["choices"][0]["message"]["content"])
    time.sleep(16)
finally:
    box.terminate(); box.wait()
print("live view:", len(live), "bytes of MPEG-TS" if live[:1] == b"\x47" else "NOT MPEG-TS")
print("/v1/chat/completions:", {k: cam[k] for k in ("state", "moving", "looks_skipped", "segments")})

segments = sorted((rec / "gate").glob("*.ts"))
frames = [int(out("ffprobe", "-v", "error", "-count_frames", "-select_streams", "v", "-show_entries", "stream=nb_read_frames",
                  "-of", "csv=p=0", str(s)).split()[0] or 0) for s in segments[:-1]]
print(f"recorded: {len(segments)} segments; frames in the closed ones: {frames}")
activity = [json.loads(l) for f in (rec / "gate" / "activity").glob("*.jsonl") for l in f.read_text().splitlines() if "from" in l]
print("activity:", activity[:3])
print("sweep (dry run):", out(str(B), "sweep", "--config", str(cfg), "--dry-run").strip()[:300])

# 3. The Lean 4 proof of the retention rules: core Lean 4.31 and omega, no Mathlib.
elan = Path.home() / ".elan/bin"
if not (elan / "lean").exists():
    sh("bash", "-c", "curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.31.0")
sh(str(elan / "lean"), "proofs/Retention.lean", cwd=CRATE)
assert "sorry" not in (CRATE / "proofs/Retention.lean").read_text()
print("proofs/Retention.lean: checked by Lean", out(str(elan / "lean"), "--version").split(",")[0])
